# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
import os

ruta = "file:" + os.getcwd() + "/ventas_export - ventas_export.csv"

bronce = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ruta)
)

bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
CREATE OR REPLACE TEMP VIEW dim_tienda AS
SELECT * FROM VALUES
    (1, 'Centro'),
    (2, 'Escalon'),
    (3, 'Soyapango')
AS (tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================================
# 1. Preparar datos originales
# ==========================================================

df = bronce

# Convertir campos necesarios
df_limpio = (
    df
    .withColumn("tienda_id_num", F.trim(F.col("tienda_id")).cast("int"))
    .withColumn(
        "monto_num",
        F.regexp_replace(
            F.regexp_replace(F.trim(F.col("monto")), " ", ""),
            ",",
            "."
        ).cast("double")
    )
)

# ==========================================================
# 2. Identificar registros que deben ir a cuarentena
# ==========================================================

df_control = (
    df_limpio
    .withColumn(
        "motivo",
        F.concat_ws(
            "; ",
            F.when(
                ~F.col("tienda_id_num").isin(1, 2, 3),
                F.lit("Tienda no permitida")
            ),
            F.when(
                F.col("monto_num").isNull(),
                F.lit("Monto no válido")
            )
        )
    )
)

# Solo los que tienen algún motivo
cuarentena = (
    df_control
    .filter(F.col("motivo") != "")
    .select(
        "venta_id",
        "fecha",
        "tienda_id",
        "producto_id",
        "cantidad",
        "monto",
        "medio_pago",
        "motivo"
    )
)

# ==========================================================
# 3. Crear el hecho limpio
# ==========================================================

fact = (
    df_control
    .filter(F.col("tienda_id_num").isin(1, 2, 3))
    .filter(F.col("monto_num").isNotNull())
    .select(
        "venta_id",
        "fecha",
        F.col("tienda_id_num").alias("tienda_id"),
        "producto_id",
        "cantidad",
        F.col("monto_num").alias("monto"),
        "medio_pago"
    )
    .dropDuplicates(["venta_id"])
)

# ==========================================================
# 4. Crear las salidas
# ==========================================================

fact.createOrReplaceTempView("oro_fact_ventas")
cuarentena.createOrReplaceTempView("cuarentena_ventas")

print("PASO: pipeline creado")
print("Filas en oro_fact_ventas:", fact.count())
print("Filas en cuarentena_ventas:", cuarentena.count())
print("Columnas del hecho:", fact.columns)

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
from pyspark.sql import functions as F

oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
    select venta_id
    from oro_fact_ventas
    group by venta_id
    having count(*) > 1
)
""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c
from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c
from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [
    c for c in oro.columns
    if c.lower() == "canal"
]

sin_motivo = "motivo" not in [
    c.lower() for c in cuarentena.columns
]

oro_inflado = oro.count() >= bronce.count()

cuarentena_vacia = cuarentena.count() == 0

ok = []

ok.append(
    prueba(
        "una fila por venta_id",
        duplicados == 0
    )
)

ok.append(
    prueba(
        "solo tiendas 1, 2 y 3",
        tiendas_malas == 0
    )
)

ok.append(
    prueba(
        "monto numérico en el hecho",
        montos_malos == 0
    )
)

ok.append(
    prueba(
        "canal no entra al hecho",
        len(columnas_de_mas) == 0
    )
)

ok.append(
    prueba(
        "cuarentena con motivo y con filas",
        (not sin_motivo) and (not cuarentena_vacia)
    )
)

ok.append(
    prueba(
        "oro menor que el archivo",
        not oro_inflado
    )
)

print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")



In [0]:
print ("actualizar git")

## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
2. ¿Qué hicieron con un monto que venía en texto?
3. ¿Por qué una tienda se quedó fuera?
4. ¿Dónde quedó la columna que no estaba en el contrato?
5. Si corren el notebook otra vez, ¿el hecho crece?

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio

In [0]:

¿Qué pasó con la misma venta llegada dos veces?
Se eliminó el duplicado usando venta_id, dejando solamente una fila por venta.

¿Qué hicieron con un monto que venía en texto?
Se convirtió el monto a número, incluyendo los casos donde venía con coma. Cuando no se pudo convertir, se envió a cuarentena.

¿Por qué una tienda se quedó fuera?
Porque solo las tiendas 1, 2 y 3 están permitidas por las reglas del negocio. Las demás se apartaron.

¿Dónde quedó la columna que no estaba en el contrato?
La columna canal quedó fuera de oro_fact_ventas, porque no forma parte de las columnas definidas para el hecho.

¿Si corren el notebook otra vez, el hecho crece?
No. El pipeline vuelve a crear las vistas y elimina duplicados por venta_id, por lo que ejecutar nuevamente no acumula las mismas ventas.